# Lab06.4 – ARM vs FPGA Performance: Single MAC vs Batch MAC

**Objective:** compare a single-operation MAC FPGA accelerator with a batch accelerator on Zybo legacy (PYNQ 3.0.1).

**Required:** `lab06.bit`, `lab06.hwh`, `lab06_batch.bit` and `lab06_batch.hwh` in this notebook directory. Generate them using the scripts in `hardware/` with Vivado 2022.2.

**Interpretation:** FPGA host-call time includes Python and AXI overhead. The batch engine additionally exposes the *internal FPGA cycle count*. This is an educational comparison with a Python loop, **not** a proof that FPGA is faster than optimized ARM C.

## Part A – One MAC per AXI transaction

**Step 1 – Load the original hardware.** Verify and program `lab06.bit`. Loading another overlay later will replace its PS–PL IP configuration.

In [ ]:
from pynq import Overlay, MMIO
from pathlib import Path
def find_overlay_pair(stem):
    # Lab06/notebooks -> Lab06/overlays; also supports classic side-by-side files.
    folders = (Path('../overlays'), Path('overlays'), Path('.'))
    for folder in folders:
        bit = folder / (stem + '.bit')
        hwh = folder / (stem + '.hwh')
        if bit.is_file() and hwh.is_file():
            return bit.resolve(), hwh.resolve()
    raise FileNotFoundError(
        'Missing ' + stem + '.bit/.hwh. Upload both into Lab06/overlays '
        '(or place both next to this notebook).')
BIT, HWH = find_overlay_pair('lab06')
acc = Overlay(str(BIT))
for name, info in sorted(acc.ip_dict.items()):
    if 'gpio' in name:
        print(name, hex(info['phys_addr']), info.get('type'))

**Step 2 – Configure AXI directions.** Locate the three GPIO IP blocks using their HWH metadata, then configure A, B and C as outputs and Y as input.

In [ ]:
def find_gpio(fragment):
    matches = [(name, info) for name, info in acc.ip_dict.items()
               if fragment in name.lower() and 'axi_gpio' in info.get('type', '').lower()]
    if len(matches) != 1:
        raise RuntimeError('Expected one GPIO containing %r; got %s' % (fragment, [n for n,_ in matches]))
    return MMIO(matches[0][1]['phys_addr'], matches[0][1].get('addr_range', 0x1000))

ab = find_gpio('operands_gpio')
c_gpio = find_gpio('c_gpio')
y_gpio = find_gpio('result_gpio')
DATA1, TRI1, DATA2, TRI2 = 0, 4, 8, 12
ab.write(TRI1, 0)
ab.write(TRI2, 0)
c_gpio.write(TRI1, 0)
y_gpio.write(TRI1, 0xffffffff)
print('AXI GPIO directions configured')

**Step 3 – Define equivalent operations.** One Python function writes each input through AXI and reads the FPGA result. The reference function performs the same arithmetic on ARM.

In [ ]:
def hw_mac(a, b, c):
    if not (0 <= a <= 255 and 0 <= b <= 255 and 0 <= c <= 65535):
        raise ValueError('Expected 8-bit A and B, 16-bit C (unsigned)')
    ab.write(DATA1, int(a))
    ab.write(DATA2, int(b))
    c_gpio.write(DATA1, int(c))
    return y_gpio.read(DATA1)

def sw_mac(a, b, c):
    return int(a) * int(b) + int(c)

**Step 4 – Verify correctness.** Execute one operation with both implementations and confirm identical output before measuring performance.

In [ ]:
a,b,c = 23,19,101
assert hw_mac(a,b,c) == sw_mac(a,b,c)
print('Verified result:', sw_mac(a,b,c))

**Step 5 – Benchmark.** Repeat 300 operations and measure average time per operation. The FPGA value includes *all AXI register transactions* and Python overhead; it is **not** the arithmetic propagation delay of `mac8`.

In [ ]:
import time
import statistics
N=300
pairs=[((i*17)%256,(i*29)%256,(i*383)%65536) for i in range(N)]
for a,b,c in pairs[:20]: hw_mac(a,b,c)
t0=time.perf_counter_ns()
arm_results=[sw_mac(a,b,c) for a,b,c in pairs]
t1=time.perf_counter_ns()
fpga_results=[hw_mac(a,b,c) for a,b,c in pairs]
t2=time.perf_counter_ns()
assert arm_results == fpga_results
arm_us=(t1-t0)/N/1000.0
roundtrip_us=(t2-t1)/N/1000.0
print('Average ARM Python computation: {:.2f} microseconds / call'.format(arm_us))
print('Average FPGA AXI round trip:   {:.2f} microseconds / call'.format(roundtrip_us))
print('AXI/ARM measured ratio: {:.2f}'.format(roundtrip_us/arm_us if arm_us else float('inf')))

**Step 6 – Visualize timing.** The bar graph shows why a small hardware operation may have a longer end-to-end latency than local Python. This does not mean the FPGA arithmetic itself is slower.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.bar(['ARM Python calculation','FPGA AXI round trip'], [arm_us,roundtrip_us])
plt.ylabel('Measured microseconds / call')
plt.title('Software computation vs hardware transaction overhead')
plt.grid(axis='y',alpha=.25)
plt.show()

**Part A conclusion:** the original design transfers three operands and reads one result for *every* MAC. Python + MMIO + AXI overhead may dominate the FPGA computation.

## Part B – Many MAC operations per FPGA launch

The new overlay generates its operands *inside* the FPGA. Both ARM and FPGA start at A=B=C=0, add 17, 53, 997 respectively each iteration, and accumulate A×B+C modulo 2³². A hardware cycle counter allows the computation phase to be measured separately.

**Step 7 – Load the batch overlay.** Reprogram the PL with `lab06_batch.bit`. Do **not** use AXI MMIO objects from Part A after this cell.

In [ ]:
from pathlib import Path
from pynq import Overlay, MMIO

batch_bit, batch_hwh = find_overlay_pair('lab06_batch')
batch_overlay = Overlay(str(batch_bit))
print('Batch overlay loaded. IP blocks:')
for name, info in sorted(batch_overlay.ip_dict.items()):
    if 'gpio' in name.lower():
        print(' ', name, 'AXI base:', hex(info['phys_addr']))
assert all(name in batch_overlay.ip_dict for name in
           ['control_gpio', 'results_gpio', 'status_gpio']), 'Unexpected batch overlay IP names'

**Step 8 – Register map.** Configure two AXI output registers (N and START), two input registers (SUM and CYCLES), and the DONE/BUSY status flags. Read the state before launching.

In [ ]:
def batch_mmio(name):
    info = batch_overlay.ip_dict[name]
    if 'axi_gpio' not in info.get('type', '').lower():
        raise RuntimeError('Not an AXI GPIO: ' + name)
    return MMIO(info['phys_addr'], info.get('addr_range', 0x1000))

control = batch_mmio('control_gpio')
results = batch_mmio('results_gpio')
status = batch_mmio('status_gpio')
DATA1, TRI1, DATA2, TRI2 = 0x00, 0x04, 0x08, 0x0C
control.write(TRI1, 0x00000000)
control.write(TRI2, 0x00000000)
results.write(TRI1, 0xFFFFFFFF)
results.write(TRI2, 0xFFFFFFFF)
status.write(TRI1, 0xFFFFFFFF)
control.write(DATA2, 0)
print('Registers: control N=0x00, START=0x08; results SUM=0x00, CYCLES=0x08; status=0x00')
print('Status before first run:', bin(status.read(DATA1) & 0x3))

**Step 9 – Reference algorithm.** Python implements the exact same sequence, arithmetic widths and 32-bit accumulation wraparound as the RTL. Inspect the small-N sums.

In [ ]:
def batch_reference(n):
    if not isinstance(n, int) or not (0 <= n <= 1_000_000):
        raise ValueError('N must be an integer in [0, 1000000]')
    a = b = c = total = 0
    for _ in range(n):
        total = (total + a * b + c) & 0xFFFFFFFF
        a = (a + 17) & 0xFF
        b = (b + 53) & 0xFF
        c = (c + 997) & 0xFFFF
    return total

for n in (0, 1, 2, 10, 100):
    print('N = {:>3} -> software sum = {}'.format(n, batch_reference(n)))

**Step 10 – FPGA command function.** Write N, pulse START (0→1→0), poll DONE with a timeout, and read the result and cycle counter. The host stopwatch stops **after the result-register reads**, so total time includes launch, polling and readback.

In [ ]:
import time

def batch_hardware(n, timeout_s=5.0):
    if not isinstance(n, int) or not (0 <= n <= 1_000_000):
        raise ValueError('N must be an integer in [0, 1000000]')
    t0 = time.perf_counter_ns()
    control.write(DATA1, n)
    control.write(DATA2, 0)
    control.write(DATA2, 1)  # rising edge, sampled by FPGA clock
    control.write(DATA2, 0)
    deadline = time.monotonic() + timeout_s
    while (status.read(DATA1) & 0x1) == 0:
        if time.monotonic() >= deadline:
            raise TimeoutError('Batch MAC DONE timeout: check overlay, GPIO connections and timing')
    hw_sum = results.read(DATA1)
    hw_cycles = results.read(DATA2)
    # Include result readbacks in the host-visible end-to-end measurement.
    elapsed_us = (time.perf_counter_ns() - t0) / 1000.0
    return hw_sum, hw_cycles, elapsed_us

**Step 11 – Validate functionality before timing.** Compare every output to the Python reference and verify that the hardware cycle count equals N. Include zero and boundary cases.

In [ ]:
for n in (0, 1, 2, 7, 100, 4096, 100000):
    expected = batch_reference(n)
    observed, cycles, wall_us = batch_hardware(n)
    print('N={:>6}: Python SUM={:>10}, FPGA SUM={:>10}, cycles={:>6}, wall={:>9.2f} us'.format(
          n, expected, observed, cycles, wall_us))
    assert observed == expected, 'Result mismatch for N={}'.format(n)
    assert cycles == n, 'Cycle-count mismatch for N={}'.format(n)
print('All batch MAC checks passed.')

**Step 12 – Benchmark three quantities.** For each N, measure Python execution on ARM, the **end-to-end FPGA service time** (including AXI writes, status polling and readbacks), and FPGA internal time from its cycle counter. The Tcl design specifies FCLK0 = 100 MHz: one cycle is 10 ns. Check frequency and timing closure in Vivado.

In [ ]:
import statistics

FCLK_HZ = 100_000_000  # hardware/build_lab06_batch.tcl: FCLK0 = 100 MHz
LENGTHS = [100, 1000, 10000, 100000]
REPEATS = 3
measurements = []
for n in LENGTHS:
    python_times = []
    fpga_roundtrips = []
    cycle_counts = []
    expected = batch_reference(n)
    for _ in range(REPEATS):
        t0 = time.perf_counter_ns()
        answer = batch_reference(n)
        t1 = time.perf_counter_ns()
        observed, cycles, wall_us = batch_hardware(n)
        assert answer == expected == observed and cycles == n
        python_times.append((t1 - t0) / 1000.0)
        fpga_roundtrips.append(wall_us)
        cycle_counts.append(cycles)
    med_python = statistics.median(python_times)
    med_roundtrip = statistics.median(fpga_roundtrips)
    internal_us = statistics.median(cycle_counts) / FCLK_HZ * 1e6
    measurements.append((n, med_python, med_roundtrip, internal_us))

print('{:>8} {:>15} {:>18} {:>18} {:>12}'.format(
      'N', 'Python (us)', 'FPGA+AXI (us)', 'FPGA only (us)', 'SW / total'))
for n, sw, wall, internal in measurements:
    print('{:8d} {:15.2f} {:18.2f} {:18.2f} {:12.2f}'.format(n,sw,wall,internal,sw/wall))

**Step 13 – Compare execution times graphically.** Use a logarithmic time axis; the internal FPGA time is calculated from hardware cycles, not the Python clock.

In [ ]:
import matplotlib.pyplot as plt
ns = [r[0] for r in measurements]
sw_times = [r[1] for r in measurements]
fpga_total = [r[2] for r in measurements]
fpga_core = [r[3] for r in measurements]
plt.figure(figsize=(9, 5))
plt.plot(ns, sw_times, 'o-', label='ARM Python (total calculation)')
plt.plot(ns, fpga_total, 's-', label='FPGA batch, including AXI + Python')
plt.plot(ns, fpga_core, '^-', label='FPGA internal cycle counter')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Number of MAC operations, N')
plt.ylabel('Elapsed time (microseconds)')
plt.title('Batch MAC on Zybo: communication versus computation')
plt.grid(True, which='both', alpha=0.25)
plt.legend()
plt.show()

## Interpreting the measured values

**Step 14 – Quantify overhead.** Define `T_overhead = T_FPGA+AXI − T_FPGA only` and `speedup = T_Python / T_FPGA+AXI`. The overhead includes Python, MMIO, AXI accesses and waiting, **not just bus transmission latency**. The `compute share` indicates what fraction of the host-observed total is spent processing in the FPGA.

In [ ]:
print('{:>8} {:>15} {:>16} {:>16}'.format('N', 'Overhead (us)', 'Compute share (%)', 'Speedup (x)'))
for n, sw, wall, internal in measurements:
    overhead = wall - internal
    share = 100.0 * internal / wall
    speedup = sw / wall
    print('{:8d} {:15.2f} {:16.1f} {:16.2f}'.format(n, overhead, share, speedup))

**Step 15 – Visualize speedup versus batch size.** A ratio above 1 means that the measured batch FPGA call was faster **than the particular Python implementation used here**. A larger batch amortizes fixed control overhead.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(ns, [sw/wall for _, sw, wall, _ in measurements], 'o-')
plt.axhline(1, color='gray', linestyle='--', linewidth=1, label='Same execution time')
plt.xscale('log')
plt.xlabel('Number of MAC operations, N')
plt.ylabel('Speedup: Python time / FPGA total time')
plt.title('Effect of batch size on observed speedup')
plt.grid(True, which='both', alpha=0.25)
plt.legend()
plt.show()

## Worked interpretation – one actual Zybo run

| N | ARM Python (µs) | FPGA + AXI (µs) | FPGA only (µs) | Python / FPGA total |
|---:|---:|---:|---:|---:|
| 100 | 485.72 | 159.05 | 1.00 | 3.05× |
| 1,000 | 4,867.26 | 165.59 | 10.00 | 29.39× |
| 10,000 | 48,755.25 | 206.13 | 100.00 | 236.53× |
| 100,000 | 474,617.20 | 1,103.38 | 1,000.00 | 430.15× |

These values were observed during a previous Zybo experiment; **they are not expected results for every execution**, especially after including result readbacks in the stopwatch measurement.

At 100 MHz, 100,000 cycles correspond to **1 ms** of core work. In that run, approximately **90.6%** of the FPGA call was internal computation. At N=100, only 1 µs of the approximately 159 µs call was internal work: fixed control overhead dominates. The system becomes more efficient as the work per launch increases.

**Limits:** the baseline is interpreted Python, not optimized ARM C. The workload internally generates input operands in FPGA; it avoids bulk transfers. Real camera-image acceleration needs explicit data movement, memory and possibly AXI DMA.

## Student questions and conclusions

1. Why can the original single MAC call be slower than a Python multiplication?
2. At 100 MHz, what is the duration of 100,000 FPGA clock cycles?
3. Explain the difference between `FPGA+AXI` and `FPGA only`. What else is included in the overhead?
4. Why does observed speedup rise as N increases?
5. Why would a fair ARM-versus-FPGA comparison also benchmark compiled C?
6. Why is generating operands inside the FPGA easier than transferring real image pixels?

**Conclusion:** accelerator performance depends on transfer granularity, control overhead and useful work per command. This experiment measures a **specific deterministic workload**, not an unconditional FPGA advantage.